**NOTICE:**
The U.S. Army Corps of Engineers, Risk Management Center (USACE-RMC) makes no guarantees about the results, or appropriateness of outputs, obtained from Numerics.


# 00. Getting Started with RMC-BestFit in Python
Welcome! This notebook will guide you through setting up and using the USACE-RMC BestFit library in Python via PythonNet.

## What You Will Learn
- Load `Numerics.dll` and `RMC.BestFit.dll`
- Import and use `RMC.BestFit` namespaces directly
- Build a BestFit `DataFrame` with `ExactData`
- Build a BestFit `UnivariateDistribution` model
- Run BestFit `MaximumLikelihood` and `BayesianAnalysis`

## Virtual Python Environment

## What is BestFit?

## Obtaining BestFit from the USACE-RMC GitHub Page

## Step 1: Load the .NET Runtime.
**CRITICAL:** You must load the .NET runtime before importing `clr`. This is the most common setup mistake.

Runtime selection:
- Use `pythonnet.load("coreclr")` for .NET 6/7/8.
- Use `pythonnet.load("netfx")` only for .NET Framework (Windows-only).

If you're not sure which you have installed, check the Numerics build output folder (e.g., `net8.0` indicates CoreCLR).


In [ ]:
import pythonnet

# Use netfx for this Beta-3 build (Windows/.NET Framework).
# Switch to coreclr if your BestFit build targets .NET 6+.
pythonnet.load("netfx")

print("OK: .NET runtime loaded")


## Step 2: Import clr and Load DLLs

Now we can import the CLR bridge and load the BestFit and Numerics assemblies. A lot of BestFit methods are built off of the Numerics library and require Numerics objects to be passed to them. To learn more about using Numerics in Python see the Numerics-Python-Examples repo on the USACE-RMC GitHub page. We recommend completing that demo before this one, but it is not required.

When you download BestFit, it should come with both the RMC.BestFit.dll and Numerics.dll under the Libraries folder. We are using the latest version of BestFit Version 2.0 (Beta-3).


In [ ]:
import clr
from pathlib import Path

bestfit_path = Path(r"C:\GIT\RMC-BestFit Version 2.0 (Beta-3)\Release\Libraries\RMC.BestFit.dll")
numerics_path = Path(r"C:\GIT\RMC-BestFit Version 2.0 (Beta-3)\Release\Libraries\Numerics.dll")

if not bestfit_path.exists():
    raise FileNotFoundError(f"Cannot find RMC.BestFit.dll at {bestfit_path}")
if not numerics_path.exists():
    raise FileNotFoundError(f"Cannot find Numerics.dll at {numerics_path}")

clr.AddReference(str(numerics_path))
clr.AddReference(str(bestfit_path))

print(f"OK: Loaded Numerics from {numerics_path}")
print(f"OK: Loaded BestFit from {bestfit_path}")


## Step 3: Import Namespaces
Now we can import classes from Numerics and BestFit just like any other Python module. Be sure to check how the namespaces are organized within each library. Below we import the namespaces needed for a basic example for the the Normal Distribution.

In [63]:
from Numerics.Distributions import UnivariateDistributionType
from Numerics.Sampling.MCMC import MCMCResults
from RMC.BestFit import DataFrame, ExactData, BayesianAnalysis
from RMC.BestFit.Model import UnivariateDistribution
from RMC.BestFit.Estimation import MaximumLikelihood, OptimizationMethod

print("OK: Imported Beta-3 BestFit namespaces successfully")


OK: Imported Beta-3 BestFit namespaces successfully


## Basic Example: GEV Analysis
First we need to create a BestFit DataFrame. This is the core BestFit input container used by analyses. Unlike simple arrays of numbers, the DataFrame supports the complex data types encountered in real-world flood frequency analysis: exact observations, uncertain measurements with error distributions, interval-censored data from paleoflood studies, and perception thresholds for historical floods.

The DataFrame supports the complex data types encountered in real-world flood frequency analysis: exact observations (`ExactData`), uncertain measurements with error distributions (`UncertainData`), interval-censored data from paleoflood studies (`IntervalData`), and perception thresholds for historical floods (`ThresholdData`). We will cover these more later on. For now, we're using exact data.

In [ ]:
annual_peaks = [
    8200, 9100, 7600, 10500, 9800,
    11200, 11900, 10100, 9600, 12400,
    13200, 11700, 10900, 12100, 11400,
]

df = DataFrame()
start_year = 2001

for i, value in enumerate(annual_peaks):
    df.ExactSeries.Add(ExactData(start_year + i, float(value)))

df.PlottingParameter = 0.0  # Weibull plotting positions
df.CalculatePlottingPositions()

# DataFrame has many properties and methods to explore the data. Here we show a few of them.
print("BestFit DataFrame summary")
print(f"- Exact observations: {df.ExactSeries.Count}")
print(f"- Total record length: {df.TotalRecordLength()}")
print(f"- Plotting parameter: {df.PlottingParameter}")

print("First 5 records")
for i in range(min(5, df.ExactSeries.Count)):
    rec = df.ExactSeries[i]
    print(f"  index={rec.Index}, value={rec.Value:.1f}, p={rec.PlottingPosition:.4f}")

### Build a BestFit Univariate Model
Create a BestFit univariate model (not just a Numerics distribution object). In RMC-BestFit, a model is any mathematical representation of a physical process that can be fitted to data. Models define the relationship between parameters and observations, specify prior distributions for Bayesian inference, and compute likelihood functions that measure how well parameter values explain the data.

We will chose to use the Generalized Extreme Value (GEV) Distribution, a pre built model commonly used in flood frequency analysis:

In [ ]:
gev_model = UnivariateDistribution(df, UnivariateDistributionType.GeneralizedExtremeValue)

print("BestFit model summary")
print(f"- Distribution type: {gev_model.Distribution.Type}")
print(f"- Display name: {gev_model.Distribution.DisplayName}")
print(f"- Number of parameters: {gev_model.NumberOfParameters}")

### Run BestFit MaximumLikelihood
Now we run a BestFit maximum-likelihood estimation directly on the `UnivariateDistribution` model.


In [ ]:
mle = MaximumLikelihood(gev_model, OptimizationMethod.NelderMead)
mle.Estimate()

print("MaximumLikelihood summary")
print(f"- Estimated: {mle.IsEstimated}")
print(f"- Number of parameters: {mle.NumberOfParameters}")

param_names = list(gev_model.Distribution.ParameterNames)
param_values = list(gev_model.Distribution.GetParameters)
print("Estimated distribution parameters")
for name, value in zip(param_names, param_values):
    print(f"- {name}: {float(value):.6g}")

x = 12000.0
print(f"CDF({x:.0f}) after MLE fit: {gev_model.Distribution.CDF(x):.6f}")


### Configure and Run BestFit BayesianAnalysis
This uses BestFit's Bayesian engine directly with the same univariate model.


In [ ]:
bayes = BayesianAnalysis(gev_model)

# Keep runtime short for a getting-started notebook.
bayes.NumberOfChains = 4
bayes.ThinningInterval = 5
bayes.WarmupIterations = 100
bayes.Iterations = 250

print("BayesianAnalysis settings")
print(f"- Chains: {bayes.NumberOfChains}")
print(f"- Thinning: {bayes.ThinningInterval}")
print(f"- Warmup: {bayes.WarmupIterations}")
print(f"- Iterations: {bayes.Iterations}")
print(f"- IsValid: {bayes.IsValid()}")


# Beta-3 note:
# bayes.RunAsync(None, True, True) throws a NullReferenceException when reporter is None.
# bayes.RunAsync(None, False, ...) may finish but clear results internally.
# Workaround: run the sampler directly and build MCMCResults yourself.

bayes.SetUpSampler()
if bayes.Sampler is None:
    raise RuntimeError("Bayesian sampler could not be initialized.")

bayes.Sampler.ParallelizeChains = True
bayes.Sampler.Sample()

mcmc_results = MCMCResults(bayes.Sampler, 1 - bayes.CredibleIntervalWidth)

print("Sampler run complete")
print(f"- Posterior sample count: {mcmc_results.Output.Count}")
print(f"- Parameter count: {mcmc_results.PosteriorMean.Values.Length}")

names = list(gev_model.Distribution.ParameterNames)
means = list(mcmc_results.PosteriorMean.Values)
print("Posterior mean parameters")
for name, value in zip(names, means):
    print(f"- {name}: {float(value):.6g}")

bayes.RunAsync(None, False, False)
print(f"Estimated: {bayes.IsEstimated} successfully")
# COME BACK SOMETHING IS BROKEN HERE....

BayesianAnalysis settings
- Chains: 4
- Thinning: 5
- Warmup: 100
- Iterations: 250
- IsValid: True
Sampler run complete
- Posterior sample count: 10000
- Parameter count: 3
Posterior mean parameters
- Location (ξ): nan
- Scale (α): nan
- Shape (κ): nan
Estimated: False successfully


## Step 7: Reflection Helpers Focused on BestFit
Use reflection to discover BestFit namespaces and classes available in your local DLL.


In [67]:
import System
from System.Reflection import Assembly

bestfit_assembly = Assembly.LoadFrom(str(bestfit_path))

all_types = [t.FullName for t in bestfit_assembly.GetTypes()]
analysis_types = [
    t for t in all_types
    if t.startswith("RMC.BestFit.") and t.endswith("Analysis")
]
model_types = [
    t for t in all_types
    if ".Model." in t
]
root_data_types = [
    t for t in all_types
    if t.startswith("RMC.BestFit.DataFrame")
    or t.startswith("RMC.BestFit.Exact")
    or t.startswith("RMC.BestFit.Interval")
    or t.startswith("RMC.BestFit.Uncertain")
    or t.startswith("RMC.BestFit.Threshold")
]

print("BestFit assembly")
print(f"- Name: {bestfit_assembly.GetName().Name}")
print(f"- Analysis types found: {len(analysis_types)}")
print(f"- Model types found: {len(model_types)}")
print(f"- Root data types found: {len(root_data_types)}")

print("Sample analysis classes")
for name in sorted(analysis_types)[:10]:
    print(f"- {name}")

print("Sample model classes")
for name in sorted(model_types)[:10]:
    print(f"- {name}")


BestFit assembly
- Name: RMC.BestFit
- Analysis types found: 13
- Model types found: 71
- Root data types found: 26
Sample analysis classes
- RMC.BestFit.BayesianAnalysis
- RMC.BestFit.BivariateAnalysis
- RMC.BestFit.CoincidentFrequencyAnalysis
- RMC.BestFit.CompositeAnalysis
- RMC.BestFit.FittingAnalysis
- RMC.BestFit.IBayesianAnalysis
- RMC.BestFit.IFrequencyAnalysis
- RMC.BestFit.MixtureAnalysis
- RMC.BestFit.PointProcessAnalysis
- RMC.BestFit.RatingCurveAnalysis
Sample model classes
- RMC.BestFit.Model.ARMA
- RMC.BestFit.Model.ARMA+<>c
- RMC.BestFit.Model.ARMAX
- RMC.BestFit.Model.ARMAX+<>c
- RMC.BestFit.Model.ARMAX+Trend
- RMC.BestFit.Model.AutoRegressive
- RMC.BestFit.Model.AutoRegressive+<>c
- RMC.BestFit.Model.BivariateDistribution
- RMC.BestFit.Model.BivariateDistribution+<>c
- RMC.BestFit.Model.CompetingRisksModel


## Common Problems and Solutions

## Summary

## Exercise
